# 02 — Silver: Hospital Dimension

**Pipeline:** Bronze → Silver  
**Source:** AIHW reporting-units API — WA hospitals with location and LHN mapping  
**Target:** `silver.dim_hospitals` (Delta table)  

The AIHW reporting-units endpoint provides:
- Hospital name and code
- Latitude / longitude
- LHN (Local Hospital Network = health service) mapping
- State mapping (pre-filtered to WA by ingest script)

In [ ]:
# ------------------------------------------------------------------
# 1. Read WA hospital JSON from bronze layer
#    Ingested by scripts/ingest_bronze.py
# ------------------------------------------------------------------
from pyspark.sql.functions import col, explode, when

raw = spark.read.option("multiline", "true").json(
    "Files/bronze/aihw/reporting_units/wa_hospitals.json"
)
print("Raw schema:")
raw.printSchema()

# Explode the result array
hospitals_raw = raw.select(explode(col("result")).alias("h")).select("h.*")
print(f"\nWA hospitals loaded: {hospitals_raw.count()}")
hospitals_raw.show(3, truncate=False)

In [ ]:
# ------------------------------------------------------------------
# 2. Extract LHN (health service) and state from nested mappings
# ------------------------------------------------------------------
from pyspark.sql.functions import filter as array_filter, transform, element_at

# mapped_reporting_units is an array of mappings
# We want to extract where map_type.mapped_reporting_unit_code == 'H_LHN'

hospitals_exploded = hospitals_raw.select(
    col("reporting_unit_code").alias("hospital_code"),
    col("reporting_unit_name").alias("hospital_name"),
    col("latitude"),
    col("longitude"),
    col("private"),
    col("closed"),
    explode(col("mapped_reporting_units")).alias("mapping")
)

# Extract LHN (health service) mapping
lhn_mapping = hospitals_exploded \
    .filter(col("mapping.map_type.mapped_reporting_unit_code") == "H_LHN") \
    .select(
        col("hospital_code"),
        col("hospital_name"),
        col("latitude"),
        col("longitude"),
        col("private"),
        col("closed"),
        col("mapping.mapped_reporting_unit.reporting_unit_name").alias("health_service")
    )

print(f"Hospitals with LHN mapping: {lhn_mapping.count()}")
lhn_mapping.show(10, truncate=False)

In [ ]:
# ------------------------------------------------------------------
# 3. Handle hospitals without LHN — add them with null health_service
# ------------------------------------------------------------------
hospital_codes_with_lhn = lhn_mapping.select("hospital_code")

hospitals_no_lhn = hospitals_raw.select(
    col("reporting_unit_code").alias("hospital_code"),
    col("reporting_unit_name").alias("hospital_name"),
    col("latitude"),
    col("longitude"),
    col("private"),
    col("closed")
).join(hospital_codes_with_lhn, "hospital_code", "left_anti") \
 .withColumn("health_service", col("closed").cast("string").cast("string"))

# Null out health_service for unmatched
from pyspark.sql.functions import lit
hospitals_no_lhn = hospitals_no_lhn.withColumn("health_service", lit(None).cast("string"))

dim_hospitals = lhn_mapping.unionByName(hospitals_no_lhn)

print(f"Total hospital dimension rows: {dim_hospitals.count()}")
print("\nHealth services:")
dim_hospitals.groupBy("health_service").count().orderBy("count", ascending=False).show()

In [ ]:
# ------------------------------------------------------------------
# 4. Validate coordinates are in WA bounds
#    WA: lon 113-130, lat -35 to -13
# ------------------------------------------------------------------
out_of_bounds = dim_hospitals.filter(
    (col("longitude").isNotNull()) &
    (
        (col("longitude") < 113) | (col("longitude") > 130) |
        (col("latitude") < -35) | (col("latitude") > -13)
    )
).count()
print(f"Hospitals outside WA bounds: {out_of_bounds}")
print(f"Hospitals with no coordinates: {dim_hospitals.filter(col('latitude').isNull()).count()}")

In [ ]:
# ------------------------------------------------------------------
# 5. Write silver Delta table
# ------------------------------------------------------------------
dim_hospitals.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.dim_hospitals")

print("silver.dim_hospitals written successfully")
print(f"Total hospitals: {spark.table('silver.dim_hospitals').count()}")